# Recomendação de Filmes Similares

In [32]:
import pandas as pd

r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, usecols=range(3), encoding="ISO-8859-1")

m_cols = ['movie_id', 'title']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols, usecols=range(2), encoding="ISO-8859-1")

ratings = pd.merge(movies, ratings)
ratings.head()

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3


* A função pivot_table em um DataFrame construirá uma matriz de avaliações de usuário/filme. Observe como NaN indica dados ausentes – ou seja, filmes que determinados usuários não avaliaram.

In [33]:
movieRatings = ratings.pivot_table(index=['user_id'],columns=['title'],values='rating')
movieRatings.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
movie_name = movieRatings['2001: A Space Odyssey (1968)']
movie_name.head()

user_id
0    NaN
1    4.0
2    NaN
3    NaN
4    NaN
Name: 2001: A Space Odyssey (1968), dtype: float64

* A função **corrwith** do Pandas facilita muito o cálculo da correlação par-a-par entre o vetor de avaliações dos usuários para *Star Wars* e todos os outros filmes! Depois disso, eliminaremos os resultados sem dados e construiremos um novo DataFrame com os filmes e seus respectivos escores de correlação (similaridade) com *Star Wars*.

In [35]:
similarMovies = movieRatings.corrwith(movie_name)
similarMovies = similarMovies.dropna()
df = pd.DataFrame(similarMovies)
df.head(10)

d:\Ferramentas\anaconda\lib\site-packages\numpy\lib\function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\Ferramentas\anaconda\lib\site-packages\numpy\lib\function_base.py:2846: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
d:\Ferramentas\anaconda\lib\site-packages\numpy\lib\function_base.py:2705: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
d:\Ferramentas\anaconda\lib\site-packages\numpy\lib\function_base.py:2705: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
d:\Ferramentas\anaconda\lib\site-packages\numpy\lib\function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,0
title,
'Til There Was You (1997),-0.426401
1-900 (1994),-0.981981
101 Dalmatians (1996),-0.043407
12 Angry Men (1957),0.178848
187 (1997),-0.554700
2 Days in the Valley (1996),0.332724
"20,000 Leagues Under the Sea (1954)",0.259308
2001: A Space Odyssey (1968),1.000000
"39 Steps, The (1935)",0.367256


* Nossos resultados provavelmente estão sendo distorcidos por filmes que foram assistidos por apenas algumas pessoas, que por acaso também gostaram de *Star Wars*. Portanto, precisamos eliminar filmes que tiveram poucas avaliações e que estão gerando resultados espúrios. Vamos construir um novo DataFrame que conte quantas avaliações existem para cada filme e também calcule a média dessas avaliações.

In [36]:
import numpy as np
movieStats = ratings.groupby('title').agg({'rating': [np.size, np.mean]})
movieStats.head()

C:\Users\Hian\AppData\Local\Temp\ipykernel_6440\225782347.py:2: FutureWarning: The provided callable <function mean at 0x0000022219DF5750> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieStats = ratings.groupby('title').agg({'rating': [np.size, np.mean]})


rating          
                            size      mean
title                                     
'Til There Was You (1997)      9  2.333333
1-900 (1994)                   5  2.600000
101 Dalmatians (1996)        109  2.908257
12 Angry Men (1957)          125  4.344000
187 (1997)                    41  3.024390

* Vamos remover todos os filmes que foram avaliados por menos de 100 pessoas e verificar quais são os mais bem avaliados que restaram.

In [37]:
popularMovies = movieStats['rating']['size'] >= 100
movieStats[popularMovies].sort_values([('rating', 'mean')], ascending=False)[:15]

rating          
                                         size      mean
title                                                  
Close Shave, A (1995)                     112  4.491071
Schindler's List (1993)                   298  4.466443
Wrong Trousers, The (1993)                118  4.466102
Casablanca (1942)                         243  4.456790
Shawshank Redemption, The (1994)          283  4.445230
Rear Window (1954)                        209  4.387560
Usual Suspects, The (1995)                267  4.385768
Star Wars (1977)                          584  4.359589
12 Angry Men (1957)                       125  4.344000
Citizen Kane (1941)                       198  4.292929
To Kill a Mockingbird (1962)              219  4.292237
One Flew Over the Cuckoo's Nest (1975)    264  4.291667
Silence of the Lambs, The (1991)          390  4.289744
North by Northwest (1959)                 179  4.284916
Godfather, The (1972)                     413  4.283293

* 100 ainda pode ser um número baixo, mas esses resultados parecem bons no que se refere a "filmes bem avaliados e conhecidos pelo público". Agora, vamos unir esses dados ao nosso conjunto original de filmes similares a *Star Wars*.

In [38]:
#df = movieStats[popularMovies].join(pd.DataFrame(similarMovies, columns=['similarity']))

# Updated for newer Pandas releases that don't allow merging between different levels; we must flatten it first now.
mappedColumnsMoviestat=movieStats[popularMovies]
mappedColumnsMoviestat.columns=[f'{i}|{j}' if j != '' else f'{i}' for i,j in mappedColumnsMoviestat.columns]
df = mappedColumnsMoviestat.join(pd.DataFrame(similarMovies, columns=['similarity']))
df.head()

,rating|size,rating|mean,similarity
title,,,
101 Dalmatians (1996),109,2.908257,-0.043407
12 Angry Men (1957),125,4.344000,0.178848
2001: A Space Odyssey (1968),259,3.969112,1.000000
Absolute Power (1997),127,3.370079,-0.241580
"Abyss, The (1989)",151,3.589404,0.089206


In [39]:
df.sort_values(['similarity'], ascending=False)[:15]

,rating|size,rating|mean,similarity
title,,,
2001: A Space Odyssey (1968),259,3.969112,1.000000
True Romance (1993),104,3.615385,0.459189
Natural Born Killers (1994),128,2.953125,0.442248
Being There (1979),116,3.905172,0.425009
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963),194,4.252577,0.392916
"Clockwork Orange, A (1971)",221,3.909502,0.388071
Citizen Kane (1941),198,4.292929,0.370413
Swingers (1996),157,3.828025,0.364147
Event Horizon (1997),127,2.574803,0.341356
